# Preprocessing — Angka, uang, satuan, tanggal, lama waktu dari teks
**Kapan dipakai:** angka ditulis manusia, misalnya `Rp1.500.000,-`, `1,5 jt`, `3.700rb`,
`2-3 juta`, `0,7 kwintal`, `12 Agu 2024`, `45516` (tanggal Excel), `2 thn 3 bln`.

**Langkah:** angka + satuan → tanggal → lama waktu → simpan. Aturan angka gaya Indonesia
yang dipakai `to_number`:
- titik diikuti **tepat 3 digit** = pemisah ribuan (`1.500.000`, `70.500`)
- koma = desimal (`1,5`)
- titik lain = desimal gaya Inggris (`2.1 juta`)

Satuan dibaca dari tabel `*_SCALE` di Konfigurasi. Satuan yang tidak ada di tabel → NaN, supaya
terlihat di laporan "gagal dibaca", lalu tambahkan satuannya ke tabel.

**Data demo:** `kotor_SINTETIS` (dibuat oleh `_alat/buat_data_kotor.py`).

In [1]:
import re

import numpy as np
import pandas as pd

## Konfigurasi

In [2]:
DATA_FOLDER = "../../../data_latihan/preprocessing"   # GANTI saat lomba
DIRTY_CSV = f"{DATA_FOLDER}/kotor_SINTETIS.csv"
KEY_CSV = f"{DATA_FOLDER}/kotor_SINTETIS_kunci.csv"   # hanya ada di data latihan

MONEY_SCALE = {"": 1, "rb": 1e3, "ribu": 1e3, "k": 1e3, "jt": 1e6, "juta": 1e6, "m": 1e9,
               "miliar": 1e9, "milyar": 1e9}
WEIGHT_SCALE = {"": 1, "kg": 1, "g": 1e-3, "gr": 1e-3, "gram": 1e-3, "ons": 0.1,
                "kwintal": 100, "ton": 1000}     # tanpa satuan dianggap kg
MONTHS = {"januari": "01", "februari": "02", "maret": "03", "april": "04", "mei": "05",
          "juni": "06", "juli": "07", "agustus": "08", "september": "09", "oktober": "10",
          "november": "11", "desember": "12", "jan": "01", "feb": "02", "mar": "03", "apr": "04",
          "jun": "06", "jul": "07", "agu": "08", "agt": "08", "sep": "09", "okt": "10",
          "nov": "11", "des": "12"}
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%d-%m-%y", "%d %m %Y"]
LESS_THAN_YEAR_MONTHS = 6       # "kurang dari 1 tahun" dianggap 6 bulan
# pola: angka [- angka2 kalau rentang] satuan
VALUE_PATTERN = r"(?P<number>\d[\d.,]*)(?:\s*-\s*(?P<number2>\d[\d.,]*))?\s*(?P<unit>[a-z]*)"

## Baca data

In [3]:
table = pd.read_csv(DIRTY_CSV, dtype=str)
print(table.shape)
table[["harga", "berat", "tanggal_daftar", "lama_kerja"]].head(10)

(3000, 10)


,harga,berat,tanggal_daftar,lama_kerja
0,"± 4,3 jt","0,712 kwintal","Minggu, 12 Juli 2020",58 bulan
1,"Rp. 1.000.000,00","57,3 Kg",2024-12-15,blm setahun
2,2.000rb,81.9kg,2 Mar 2021,1 tahun 1 bulan
3,Rp 8.200.000,551 ons,17-01-19,6 tahun 9 bulan
4,Rp 1.300.000,68800 gram,2021-04-04,2 thn 2 bln
5,1-3 juta,"0,813 kwintal","Jumat, 4 September 2020",83 bln
6,Rp 3.000.000,55.800 gr,09/05/2023,2 tahun
7,"Rp. 2.100.000,00",101.500 gr,30 Mar 2021,4th2bl
8,"12,3 jt",98.8kg,27-06-24,6 tahun
9,"± 9,5 jt","103,9",01/03/2023,8th1bl


## Angka + satuan (uang dan berat)
Rentang `2-3 juta` diambil nilai tengahnya.

In [4]:
def to_number(text):
    # '1.500.000' -> 1500000, '70.500' -> 70500, '1,5' -> 1.5, '2.1' -> 2.1
    text = text.strip(".,")
    if re.fullmatch(r"\d{1,3}(\.\d{3})+(,\d+)?", text):
        text = text.replace(".", "")
    return float(text.replace(",", "."))


def read_value(text, scale):
    found = re.search(VALUE_PATTERN, str(text).lower())
    if found is None:
        return np.nan
    value = to_number(found["number"])
    if found["number2"]:
        value = (value + to_number(found["number2"])) / 2
    return value * scale.get(found["unit"], np.nan)


for text in ["Rp1.500.000,-", "± 4,3 jt", "3.700rb", "4-6 juta"]:
    print(f"{text:15s} -> {read_value(text, MONEY_SCALE):,.1f}")
for text in ["51.600 gr", "0,49 kwintal"]:
    print(f"{text:15s} -> {read_value(text, WEIGHT_SCALE):,.3f}")

Rp1.500.000,-   -> 1,500,000.0
± 4,3 jt        -> 4,300,000.0
3.700rb         -> 3,700,000.0
4-6 juta        -> 5,000,000.0
51.600 gr       -> 51.600
0,49 kwintal    -> 49.000


In [5]:
prices = []
for text in table["harga"]:
    prices.append(read_value(text, MONEY_SCALE))
price_rp = pd.Series(prices, index=table.index)

weights = []
for text in table["berat"]:
    weights.append(read_value(text, WEIGHT_SCALE))
weight_kg = pd.Series(weights, index=table.index)

print("harga gagal dibaca:", price_rp.isna().sum())
print(table.loc[price_rp.isna(), "harga"].unique()[:5])
print("berat gagal dibaca:", weight_kg.isna().sum())
print(table.loc[weight_kg.isna(), "berat"].unique()[:5])
pd.DataFrame({"harga": table["harga"], "harga_rp": price_rp, "berat": table["berat"],
              "berat_kg": weight_kg}).head(10)

harga gagal dibaca: 291
['-' '?' 'tidak tahu']
berat gagal dibaca: 0
[]


,harga,harga_rp,berat,berat_kg
0,"± 4,3 jt",4300000.0,"0,712 kwintal",71.2
1,"Rp. 1.000.000,00",1000000.0,"57,3 Kg",57.3
2,2.000rb,2000000.0,81.9kg,81.9
3,Rp 8.200.000,8200000.0,551 ons,55.1
4,Rp 1.300.000,1300000.0,68800 gram,68.8
5,1-3 juta,2000000.0,"0,813 kwintal",81.3
6,Rp 3.000.000,3000000.0,55.800 gr,55.8
7,"Rp. 2.100.000,00",2100000.0,101.500 gr,101.5
8,"12,3 jt",12300000.0,98.8kg,98.8
9,"± 9,5 jt",9500000.0,"103,9",103.9


## Tanggal
1. Buang nama hari (`Rabu, `).
2. Ganti nama bulan Indonesia dengan angka (`Agu` → `08`).
3. Coba tiap format di `DATE_FORMATS` **berurutan**: baris yang gagal di format pertama dicoba
   dengan format berikutnya.
4. Angka 5 digit dibaca sebagai tanggal Excel (hari sejak 1899-12-30).

Hati-hati `05/04/2021`: format Indonesia adalah hari/bulan (5 April), bukan bulan/hari.

In [6]:
date_text = table["tanggal_daftar"].str.lower()
date_text = date_text.str.replace(r"^[a-z]+,\s*", "", regex=True)
for name, number in MONTHS.items():
    date_text = date_text.str.replace(rf"\b{name}\b", number, regex=True)

dates = pd.Series(pd.NaT, index=table.index)
for date_format in DATE_FORMATS:
    parsed = pd.to_datetime(date_text, format=date_format, errors="coerce")
    dates = dates.fillna(parsed)

is_serial = date_text.str.fullmatch(r"\d{5}")
serial = pd.to_numeric(date_text.where(is_serial))
dates = dates.fillna(pd.to_datetime(serial, unit="D", origin="1899-12-30"))

print("tanggal gagal dibaca:", dates.isna().sum())
print(table.loc[dates.isna(), "tanggal_daftar"].head())
pd.DataFrame({"mentah": table["tanggal_daftar"], "tanggal": dates}).head(10)

tanggal gagal dibaca: 0
Series([], Name: tanggal_daftar, dtype: object)


,mentah,tanggal
0,"Minggu, 12 Juli 2020",2020-07-12
1,2024-12-15,2024-12-15
2,2 Mar 2021,2021-03-02
3,17-01-19,2019-01-17
4,2021-04-04,2021-04-04
5,"Jumat, 4 September 2020",2020-09-04
6,09/05/2023,2023-05-09
7,30 Mar 2021,2021-03-30
8,27-06-24,2024-06-27
9,01/03/2023,2023-03-01


Fitur turunan dari tanggal yang biasa dipakai model.

In [7]:
date_features = pd.DataFrame()
date_features["tahun"] = dates.dt.year
date_features["bulan"] = dates.dt.month
date_features["hari_ke"] = dates.dt.dayofweek          # 0 = Senin
date_features["akhir_pekan"] = dates.dt.dayofweek >= 5
date_features["umur_hari"] = (pd.Timestamp("2025-01-01") - dates).dt.days
date_features.head()

,tahun,bulan,hari_ke,akhir_pekan,umur_hari
0,2020,7,6,True,1634
1,2024,12,6,True,17
2,2021,3,1,False,1401
3,2019,1,3,False,2176
4,2021,4,6,True,1368


## Lama waktu → bulan
Ambil bagian tahun dan bagian bulan terpisah. Bentuk rapat seperti `1th2bl` juga tertangkap
karena pola tidak memakai `\b`.

In [8]:
duration = table["lama_kerja"].str.lower()
years = duration.str.extract(r"(\d+(?:,\d+)?)\s*(?:tahun|thn|th)")[0]
years = pd.to_numeric(years.str.replace(",", "."))
months = pd.to_numeric(duration.str.extract(r"(\d+)\s*(?:bulan|bln|bl)")[0])

duration_months = years.fillna(0) * 12 + months.fillna(0)
duration_months = duration_months.where(years.notna() | months.notna())   # dua-duanya kosong → NaN
is_less_than_year = duration.str.contains(r"kurang|<|blm|belum")
duration_months = duration_months.mask(is_less_than_year, LESS_THAN_YEAR_MONTHS)

print("lama gagal dibaca:", duration_months.isna().sum())
pd.DataFrame({"mentah": table["lama_kerja"], "bulan": duration_months}).head(10)

lama gagal dibaca: 0


,mentah,bulan
0,58 bulan,58.0
1,blm setahun,6.0
2,1 tahun 1 bulan,13.0
3,6 tahun 9 bulan,81.0
4,2 thn 2 bln,26.0
5,83 bln,83.0
6,2 tahun,24.0
7,4th2bl,50.0
8,6 tahun,72.0
9,8th1bl,97.0


## Simpan hasil

In [9]:
result = pd.DataFrame()
result["id"] = table["id"]
result["harga_rp"] = price_rp
result["berat_kg"] = weight_kg
result["tanggal_daftar"] = dates.dt.strftime("%Y-%m-%d")
result["lama_bulan"] = duration_months
result = pd.concat([result, date_features], axis=1)
result.to_csv("hasil_angka.csv", index=False)
result.describe()

,harga_rp,berat_kg,lama_bulan,tahun,bulan,hari_ke,umur_hari
count,2.709000e+03,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,4.549022e+06,75.100967,60.451333,2021.497000,6.486000,2.950667,1098.940667
std,4.148113e+06,20.311484,34.376836,1.686299,3.398282,1.998057,622.828341
min,2.000000e+05,40.000000,1.000000,2019.000000,1.000000,0.000000,3.000000
25%,2.000000e+06,57.275000,31.000000,2020.000000,4.000000,1.000000,573.750000
50%,3.300000e+06,75.300000,61.000000,2022.000000,6.000000,3.000000,1090.000000
75%,5.700000e+06,92.600000,90.000000,2023.000000,9.000000,5.000000,1639.000000
max,4.340000e+07,110.000000,120.000000,2024.000000,12.000000,6.000000,2192.000000


## Cek dengan kunci jawaban latihan (HAPUS saat lomba)

In [10]:
key = pd.read_csv(KEY_CSV)
for column in ["harga_rp", "berat_kg", "lama_bulan"]:
    both_empty = result[column].isna() & key[column].isna()
    same = np.isclose(result[column], key[column]) | both_empty
    print(f"{column:15s} benar {same.mean():.3f}")
same = result["tanggal_daftar"] == key["tanggal_daftar"]
print(f"{'tanggal_daftar':15s} benar {same.mean():.3f}")

harga_rp        benar 1.000
berat_kg        benar 1.000
lama_bulan      benar 1.000
tanggal_daftar  benar 1.000
